# 09 — Existing Apache AGE relationships and documents
Read-only lab: no new nodes or edges are created.

Current real graph edges are `HAS_MODULE` (Course → Module), `HAS_SOURCE` (Module → Source), and `USES_TECHNOLOGY` (Course → Technology).

In [ ]:
from course_knowledge.database import connect
from course_knowledge.settings import database_settings
db = connect(database_settings())
with db.cursor() as cur:
    cur.execute('SET search_path = ag_catalog, "$user", public')
    cur.execute("SELECT * FROM cypher('course_graph', $$ MATCH ()-[r]->() RETURN DISTINCT label(r) $$) AS (relationship agtype)")
    for row in cur.fetchall(): print(row['relationship'])

## Fetch all documents in one course
This Cypher path follows Course → Module → Source/Document.

In [ ]:
course_key = 'gh-300'  # Try openai-courseware
query = f"MATCH (c:Course {{course_key: '{course_key}'}})-[:HAS_MODULE]->(m:Module)-[:HAS_SOURCE]->(s:Source) RETURN c.title, m.title, s.filename ORDER BY m.title, s.filename"
with db.cursor() as cur:
    cur.execute("SELECT * FROM cypher('course_graph', $$ " + query + " $$) AS (course agtype, module agtype, document agtype)")
    for row in cur.fetchall(): print(row)

## Fetch documents linked to a technology
The graph joins Course → Technology and Course → Module → Source.

In [ ]:
technology = 'MCP'
query = f"MATCH (c:Course)-[:USES_TECHNOLOGY]->(t:Technology {{name: '{technology}'}}), (c)-[:HAS_MODULE]->(m:Module)-[:HAS_SOURCE]->(s:Source) RETURN c.title, t.name, m.title, s.filename LIMIT 20"
with db.cursor() as cur:
    cur.execute("SELECT * FROM cypher('course_graph', $$ " + query + " $$) AS (course agtype, technology agtype, module agtype, document agtype)")
    for row in cur.fetchall(): print(row)

## Exercise
Change the course and technology values. Compare this structural graph retrieval with vector retrieval: AGE returns connected documents; pgvector returns semantically similar chunks. Factual answers must still cite chunks.